In [ ]:
# Configuración COSMIC v0.0.1 - Estructura Modular
import sys
from pathlib import Path

# Configuración automática de rutas relativas
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parents[2]  # Tres niveles arriba desde data/test/NGC6383/

# Agregar al path si no está
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"COSMIC v0.0.1 - NGC6383 Analysis")
print(f"Directorio actual: {CURRENT_DIR}")
print(f"Proyecto: {PROJECT_ROOT}")

# Verificar instalación de COSMIC
try:
    import cosmic
    print("COSMIC modular disponible")
except ImportError:
    print("Instalando COSMIC...")
    import os
    os.system(f"pip install -e {PROJECT_ROOT}")
    import cosmic
    print("COSMIC instalado")

In [ ]:
# Imports usando la nueva estructura modular
from cosmic.analysis.analyzer import ClusterAnalyzer

# El import legacy sigue funcionando para compatibilidad:
# from COSMIC import ClusterAnalyzer  # Funciona igual

print(f"ClusterAnalyzer importado: {ClusterAnalyzer}")
print(f"Listo para análisis con nueva API modular")

In [ ]:
# Configuración de datos usando rutas relativas
data_folder = 'data/40/'
file_path = data_folder + 'clustering_results.dill'

print(f"Carpeta de datos: {data_folder}")
print(f"Archivo: {file_path}")
print(f"Existe: {Path(file_path).exists()}")

# Inicializar el analizador con la nueva API
ca = ClusterAnalyzer(file_path)
print("ClusterAnalyzer inicializado correctamente")

In [ ]:
# 3) Build (or retrieve) a summary table: one row per cluster, 
#    with columns for label, n_members, persistence, etc.
ca.clusters_summary(include_noise=True)

In [ ]:
ca.plot_persistence_vs_members(percentile=0.8, figsize=(10,10))

In [ ]:
cluster_id = 12
cluster_data = ca.select_cluster(cluster_id)  # esto devuelve solo los objetos de ese cluster
print(f"Cluster {cluster_id} selected, contains {len(cluster_data)} sources.")

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8)) # a esto no se la ha hecho nada, es el cluster tal cual como slió del preprocessing.

In [ ]:
pmin = 0.5
pre = (ca.data['probability_hdbscan'] >= pmin)

In [ ]:
ca.sigma_clip_parallax(
    sigma=2.0,
    use_biweight=True,
    preselector_mask=pre,
    print_results=True,
    in_place=True
)

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8))

In [ ]:
ca.pms_characterization(
)

In [ ]:
ca.plot_pms(
    cluster=cluster_id,
    pms_threshold=0.6,    # ajusta si deseas un corte distinto
    figsize=(7,7),
    layout="tight",
)

In [ ]:
cluster_data = ca.select_cluster(cluster_id)

In [ ]:
# `loga_range`, `dm_mu` y `dm_range` son OBLIGATORIOS: son del cumulo que ajustas.
# Los de abajo son de NGC 6383.
fitter = ca.prepare_isochrone_fitter(
    cluster=cluster_id,
    isochs_path="./MIST/UBVRIplus/",
    loga_range=(6.0, 7.0),
    dm_mu=10.2,
    dm_range=(9.5, 10.7),
)

In [ ]:
# 2. Actualiza los priors ANTES de build_grid
fitter.set_priors({
    "dm_mu":      10.204,
    "dm_sigma":   0.05,
    "dm_range":   (9.95, 10.45),
    "Av_range":   (0.0, 1.5),
    "loga_range": (6.0, 6.5),
})

In [ ]:
# 3. Construye la grilla (usa los priors actualizados para el rango)
fitter.build_grid(
    M_met=15,
    M_loga=107,
    grid_cache="./data/40/hgrid_v3.npz",
)

In [ ]:
# 4. Fit
idata = fitter.fit(
    draws=5000,
    tune=2000,
    nuts_sampler="blackjax",
    initvals={"met": 0.008033, "dm": 10.204, "Av": 0.3},
)

In [ ]:
import arviz as az
az.summary(idata, var_names=["met", "loga", "dm", "Av"], round_to=4)

In [ ]:
divs = idata.sample_stats["diverging"].values.sum()
print(f"Divergencias: {divs}")
print(f"Draws totales: {idata.posterior.dims}")
print(f"\nR-hat por parámetro:")
for var in ["met", "loga", "dm", "Av", "log_s", "bg"]:
    try:
        rhat = az.rhat(idata)[var].values.item()
        ess  = az.ess(idata)[var].values.item()
        print(f"  {var:8s}  r_hat={rhat:.4f}  ess={ess:.0f}")
    except Exception as e:
        print(f"  {var}: {e}")


In [ ]:
az.plot_trace(idata, var_names=["met", "loga", "dm", "Av", "log_s", "bg"]);


In [ ]:
az.plot_energy(idata);

In [ ]:
az.plot_pair(
    idata,
    var_names=["met", "loga", "dm", "Av"],
    kind="kde",
    divergences=True,
    marginals=True,
    figsize=(10, 10),
);


In [ ]:
print(fitter._isochs.met_age_dict)
print(f"met_min = {fitter._met_grid[0]:.6f}, met_max = {fitter._met_grid[-1]:.6f}")
print(f"Unique Hess diagrams: {len(set(map(tuple, fitter._H_grid.reshape(fitter._H_grid.shape[0]*fitter._H_grid.shape[1], -1))))}")


In [ ]:
print(f"dm_mu    = {fitter.dm_mu}")
print(f"dm_sigma = {fitter.dm_sigma}")
print(f"dm_range = {fitter.dm_range}")
print(f"Av_range = {fitter.Av_range}")
print(f"loga_range = {fitter.loga_range}")


In [ ]:
fitter.plot_cmd(idata, num_samples=30)

In [ ]:
importlib.reload(_iso_mod)
fitter.plot_hess = types.MethodType(_iso_mod.IsochroneFitter.plot_hess, fitter)

fitter.plot_hess(idata)
